In [43]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["EXCHANGE_RATE_API_KEY"] = os.getenv("EXCHANGE_RATE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

EXCHANGE_RATE_API_KEY = os.environ.get("EXCHANGE_RATE_API_KEY")

In [44]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool,InjectedToolArg
import requests
from typing import Annotated

In [45]:
@tool
def get_currency_conversion_factor(baseCurrency:str, targetCurrency:str)->float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """

    url=f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/{baseCurrency}/{targetCurrency}"

    response = requests.get(url)

    return response.json()["conversion_rate"] 


@tool
def converter(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value and conversion factor
    """

    return base_currency_value * conversion_rate

In [72]:
llm = ChatGroq(model="qwen/qwen3-32b")

llm_with_tools = llm.bind_tools([get_currency_conversion_factor, converter])


In [81]:
messages=[]

query = HumanMessage("What is the conversion factor between INR and USD, and based on that can you convert 35 USD to INR")

messages.append(query)

ai_message = llm_with_tools.invoke(messages)

messages.append(ai_message)


In [82]:
ai_message.tool_calls

[{'name': 'get_currency_conversion_factor',
  'args': {'baseCurrency': 'USD', 'targetCurrency': 'INR'},
  'id': 'kxraqs144',
  'type': 'tool_call'},
 {'name': 'converter',
  'args': {'base_currency_value': 35},
  'id': 'jexttej63',
  'type': 'tool_call'}]

In [83]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 35 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s tackle the user\'s query step by step. They want two things: the conversion factor between INR and USD, and then to convert 35 USD to INR.\n\nFirst, I need to get the conversion factor. The user mentioned "conversion factor between INR and USD," which typically means 1 USD to INR. So I\'ll use the get_currency_conversion_factor function with baseCurrency as USD and targetCurrency as INR. That should give me the rate, like how many INR one USD is worth.\n\nOnce I have that rate, the next part is converting 35 USD to INR. For this, I\'ll use the converter function. But wait, the converter function\'s parameters are a bit unclear. The description says it takes a base_currency_value and a conversion rate. The user provided 35 USD, so base_currency_

In [84]:
import json
for tool_call in ai_message.tool_calls:
    if tool_call['name']=='get_currency_conversion_factor':
        toolMessage1 = get_currency_conversion_factor.invoke(tool_call)
        conversion_factor=json.loads(toolMessage1.content)
        messages.append(toolMessage1)

    if tool_call['name']=='converter':
        tool_call['args']['conversion_rate']=conversion_factor
        toolMessage2=converter.invoke(tool_call)
        messages.append(toolMessage2)
    
messages


[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 35 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s tackle the user\'s query step by step. They want two things: the conversion factor between INR and USD, and then to convert 35 USD to INR.\n\nFirst, I need to get the conversion factor. The user mentioned "conversion factor between INR and USD," which typically means 1 USD to INR. So I\'ll use the get_currency_conversion_factor function with baseCurrency as USD and targetCurrency as INR. That should give me the rate, like how many INR one USD is worth.\n\nOnce I have that rate, the next part is converting 35 USD to INR. For this, I\'ll use the converter function. But wait, the converter function\'s parameters are a bit unclear. The description says it takes a base_currency_value and a conversion rate. The user provided 35 USD, so base_currency_

In [85]:
final_response = llm_with_tools.invoke(messages)

In [86]:
final_response

AIMessage(content='The conversion factor from USD to INR is **92.9224**. \n\n35 USD is equivalent to **3,252.28 INR** (calculated as 35 × 92.9224).', additional_kwargs={'reasoning_content': 'Okay, let me figure out how to approach this. The user asked for the conversion factor between INR and USD and then to convert 35 USD to INR.\n\nFirst, I need to get the conversion factor. The function get_currency_conversion_factor requires the base and target currencies. Since they want INR to USD, but the example might have it the other way. Wait, the user mentioned "conversion factor between INR and USD", but when converting 35 USD to INR, the base currency is USD. So maybe the first step is to get the conversion factor from USD to INR. The first tool call uses baseCurrency as USD and targetCurrency as INR, which makes sense because 1 USD equals X INR. The response was 92.9224, which is the current exchange rate.\n\nNext, using the converter function. The user wants to convert 35 USD to INR. Th